# BLOK · Partie 2 · Quels devis vont se transformer ?

Objectif : un score de conversion pour prioriser les relances.
Principes : cible sans ambiguïté, variables connues le jour de l'envoi du devis, validation chronologique, comparaison honnête à des règles naïves.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, brier_score_loss

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
pd.set_option("display.float_format", lambda x: f"{x:,.0f}".replace(",", " ") if abs(x) >= 1000 else f"{x:.3f}")

con = duckdb.connect()
for t in ["organizations", "product_variants", "quotes", "quote_items", "orders"]:
    con.execute(f"CREATE TABLE {t} AS SELECT * FROM read_csv_auto('{(ROOT / 'data' / (t + '.csv')).as_posix()}')")
con.execute((ROOT / "sql" / "02_vue_devis_retenus.sql").read_text(encoding="utf-8"))
con.execute((ROOT / "sql" / "05_vue_features_devis.sql").read_text(encoding="utf-8"))
feat = con.sql("SELECT * FROM features_devis").df()

## 1. Cible et population

Cible : un devis est converti s'il existe une commande non annulée et non supprimée avec son `quote_number` (pas le statut `accepted`, voir A10).

Le piège des devis récents : un devis envoyé le 28 juin et non converti au 30 juin n'est pas perdu, il n'a pas encore eu le temps. Comme toutes les conversions arrivent au plus 12 jours après l'envoi, on n'apprend que sur les devis envoyés au plus tard le 15 juin : 352 devis, conformes aux « environ 350 » de l'énoncé. Les devis envoyés du 16 au 30 juin sont ceux à scorer. Les devis non envoyés au 30 juin (brouillons, ou date d'envoi en juillet) n'ont pas de score.

In [2]:
conv = feat[feat.converti]
print("Délai max entre envoi et commande :", (conv.date_commande - conv.sent_date).dt.days.max(), "jours")
feat.groupby("population").agg(nb_devis=("quote_number", "size"), taux_conversion=("converti", "mean"))

Délai max entre envoi et commande : 12 jours


,nb_devis,taux_conversion
population,,
issue connue : apprentissage,352,0.560
non envoyé au 30/06,23,0.000
récent : à scorer,20,0.400


## 2. Variables

Moment de la prédiction : le jour de l'envoi de la dernière version, quand le commercial décide s'il relancera.

| Variable | Connue à l'envoi ? | Pourquoi elle peut jouer |
|---|---|---|
| Montant HT (log) | Oui | Un gros devis engage plus le client |
| Nombre de lignes, catégorie principale | Oui | Nature du besoin |
| Nombre de versions | Oui (risque faible : 4 versions émises après accord oral) | Négociation en cours |
| Délai entre création et envoi, et indicateur « plus de 5 jours » | Oui | Réactivité perçue par le client |
| Commandes passées du client avant le devis | Oui, calculé point-in-time | Fidélité |
| Segment, commercial | Oui | Profil client, pratique commerciale |
| Statut du devis | Non : c'est la réponse du client | Exclu (section 4) |
| Commande, factures, CA total du client sur la période | Non : postérieurs ou incluant le futur | Exclus |
| Date de création du client | Non fiable (A17) | Exclue |

In [3]:
feat["log_montant"] = np.log1p(feat.montant_ht)
feat["envoi_tardif"] = (feat.delai_envoi_j > 5).astype(float)
NUM = ["log_montant", "nb_lignes", "nb_versions", "delai_envoi_j", "envoi_tardif", "client_nb_commandes_avant"]
CAT = ["segment", "categorie_principale", "salesman"]

data = feat[feat.population == "issue connue : apprentissage"].sort_values(["sent_date", "quote_number"]).reset_index(drop=True)
y = data.converti.astype(int).to_numpy()
coupure = int(len(data) * 0.75)
train, test, y_train, y_test = data.iloc[:coupure], data.iloc[coupure:], y[:coupure], y[coupure:]
print(f"Apprentissage : {len(train)} devis jusqu'au {train.sent_date.max():%d/%m/%Y} | "
      f"Test : {len(test)} devis à partir du {test.sent_date.min():%d/%m/%Y} | conversion test {y_test.mean():.0%}")

Apprentissage : 264 devis jusqu'au 18/04/2026 | Test : 88 devis à partir du 19/04/2026 | conversion test 60%


## 3. Modèle et validation

Régression logistique régularisée. Le paramètre `C` est choisi par validation croisée chronologique sur l'apprentissage seulement (on apprend sur le passé, on évalue sur le mois suivant).

In [4]:
def modele(C):
    prep = ColumnTransformer([("num", StandardScaler(), NUM), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    return make_pipeline(prep, LogisticRegression(C=C, max_iter=5000))

cv = {}
for C in [0.01, 0.03, 0.1, 0.3, 1.0]:
    aucs = [roc_auc_score(y_train[v], modele(C).fit(train.iloc[a], y_train[a]).predict_proba(train.iloc[v])[:, 1])
            for a, v in TimeSeriesSplit(n_splits=4).split(train)]
    cv[C] = np.mean(aucs)
C_RETENU = max(cv, key=cv.get)
print({c: round(float(s), 3) for c, s in cv.items()}, "-> C retenu :", C_RETENU)

{0.01: 0.594, 0.03: 0.594, 0.1: 0.58, 0.3: 0.565, 1.0: 0.559} -> C retenu : 0.03


Évaluation unique sur le test, avec intervalle de confiance par bootstrap, face à trois règles naïves :

In [5]:
def ic95(y_vrai, score, n=2000):
    rng, res = np.random.default_rng(42), []
    while len(res) < n:
        i = rng.integers(0, len(y_vrai), len(y_vrai))
        if y_vrai[i].min() != y_vrai[i].max():
            res.append(roc_auc_score(y_vrai[i], score[i]))
    return np.percentile(res, [2.5, 97.5]).round(2)

def conversion_top30(score):
    return y_test[np.argsort(-score, kind="stable")[: int(np.ceil(0.3 * len(score)))]].mean()

p_modele = modele(C_RETENU).fit(train, y_train).predict_proba(test)[:, 1]
scores = {
    "Modèle (régression logistique)":   p_modele,
    "Règle : envoyé le plus vite":      -test.delai_envoi_j.to_numpy(float),
    "Règle : client a déjà commandé":   (test.client_nb_commandes_avant > 0).to_numpy(float),
}
pd.DataFrame([{"approche": k, "auc_test": roc_auc_score(y_test, s), "ic95": ic95(y_test, s),
               "conversion_top30": conversion_top30(s)} for k, s in scores.items()]
             + [{"approche": "Aucun tri (taux moyen)", "auc_test": 0.5, "conversion_top30": y_test.mean()}])

,approche,auc_test,ic95,conversion_top30
0,Modèle (régression logistique),0.636,"[0.52, 0.75]",0.667
1,Règle : envoyé le plus vite,0.581,"[0.45, 0.71]",0.630
2,Règle : client a déjà commandé,0.510,"[0.46, 0.56]",0.481
3,Aucun tri (taux moyen),0.500,NaN,0.602


In [6]:
print("Brier modèle :", round(brier_score_loss(y_test, p_modele), 3),
      "| Brier constante :", round(brier_score_loss(y_test, np.full(len(y_test), y_train.mean())), 3))

rng, gains = np.random.default_rng(42), []
regle = scores["Règle : envoyé le plus vite"]
while len(gains) < 2000:
    i = rng.integers(0, len(y_test), len(y_test))
    if y_test[i].min() != y_test[i].max():
        gains.append(roc_auc_score(y_test[i], p_modele[i]) - roc_auc_score(y_test[i], regle[i]))
print(f"Gain d'AUC : {np.mean(gains):+.3f}, IC 95 % [{np.percentile(gains, 2.5):+.3f} ; {np.percentile(gains, 97.5):+.3f}], "
      f"positif dans {np.mean(np.array(gains) > 0):.0%} des tirages")

Brier modèle : 0.217 | Brier constante : 0.243


Gain d'AUC : +0.052, IC 95 % [-0.063 ; +0.174], positif dans 80% des tirages


Lecture.
- Le modèle obtient une AUC de 0,64 [0,52 ; 0,75] sur les 88 devis les plus récents, contre 0,58 pour la meilleure règle naïve (« envoyé le plus vite ») et 0,51 pour « client a déjà commandé ». Relancer le top 30 % des devis donne 67 % de conversion contre 60 % en moyenne. Ses probabilités sont aussi meilleures qu'une probabilité constante (Brier 0,217 contre 0,243).
- Ce gain reste fragile. L'avantage sur la règle naïve est de +0,05 d'AUC, mais l'intervalle va de -0,06 à +0,17. C'est probable, pas démontré, sur 88 devis.
- Le paramètre de régularisation a été choisi sur l'apprentissage seulement. Le test n'a servi qu'une fois : choisir un modèle d'après le test fausserait l'évaluation.

## 4. Ce que donne une variable qui contient la réponse

In [7]:
CAT_FUITE = CAT + ["status"]
prep = ColumnTransformer([("num", StandardScaler(), NUM), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FUITE)])
fuite = make_pipeline(prep, LogisticRegression(C=C_RETENU, max_iter=5000)).fit(train, y_train)
print("AUC test avec 'status' :", round(roc_auc_score(y_test, fuite.predict_proba(test)[:, 1]), 3))

AUC test avec 'status' : 0.96


L'AUC monte presque à 1, mais ce modèle est inutilisable : quand on veut prioriser les relances, le devis est au statut `sent` et le client n'a pas encore répondu. `accepted` ou `rejected`, c'est la réponse.

## 5. Interprétation et scoring

In [8]:
final = modele(C_RETENU).fit(data, y)
noms = [n.split("__")[1] for n in final[0].get_feature_names_out()]
pd.Series(final[1].coef_[0], index=noms).sort_values().to_frame("coefficient (variables standardisées)")

,coefficient (variables standardisées)
envoi_tardif,-0.266
delai_envoi_j,-0.193
log_montant,-0.183
salesman_S03,-0.127
segment_TPE,-0.088
categorie_principale_EPI & Sécurité,-0.087
categorie_principale_Alimentaire & Horeca,-0.085
nb_lignes,-0.042
segment_PME,-0.034
salesman_S02,-0.018


In [9]:
ouverts = feat[(feat.population == "récent : à scorer") & ~feat.converti].copy()
ouverts["proba"] = final.predict_proba(ouverts)[:, 1]
ouverts["valeur_attendue"] = ouverts.proba * ouverts.montant_ht
liste = ouverts.sort_values("valeur_attendue", ascending=False)[
    ["quote_number", "salesman", "sent_date", "montant_ht", "delai_envoi_j", "proba", "valeur_attendue", "status"]]
(ROOT / "outputs").mkdir(exist_ok=True)
liste.to_csv(ROOT / "outputs" / "scores_devis_ouverts_30_06.csv", index=False)
liste

,quote_number,salesman,sent_date,montant_ht,delai_envoi_j,proba,valeur_attendue,status
142,DV-2026-0404,S04,2026-06-23,5 122 800,4,0.601,3 076 604,rejected
392,DV-2026-0338,S01,2026-06-21,5 115 540,12,0.313,1 601 622,rejected
138,DV-2026-0386,S01,2026-06-29,5 263 550,7,0.273,1 434 658,sent
370,DV-2026-0280,S04,2026-06-19,1 884 740,2,0.689,1 298 720,rejected
278,DV-2026-0247,S02,2026-06-26,1 045 970,3,0.588,615 053,rejected
101,DV-2026-0351,S03,2026-06-17,1 270 510,6,0.269,341 762,sent
283,DV-2026-0411,S04,2026-06-28,648 720,6,0.406,263 425,rejected
296,DV-2026-0287,S04,2026-06-25,160 540,2,0.700,112 376,accepted
354,DV-2026-0251,S03,2026-06-25,148 600,1,0.621,92 239,sent
275,DV-2026-0151,S04,2026-06-26,110 690,13,0.437,48 365,sent


In [10]:
non_envoyes = feat[feat.population == "non envoyé au 30/06"]
sans_reponse = data[(data.status == "sent") & ~data.converti]
print(f"Non envoyés au 30/06 : {len(non_envoyes)} devis, {non_envoyes.montant_ht.sum() / 1e6:.1f} M HT")
print(f"Envoyés il y a plus de 15 jours, toujours sans réponse : {len(sans_reponse)} devis, {sans_reponse.montant_ht.sum() / 1e6:.1f} M HT")

Non envoyés au 30/06 : 23 devis, 41.5 M HT
Envoyés il y a plus de 15 jours, toujours sans réponse : 80 devis, 128.4 M HT


Lecture.
- Les variables qui pèsent le plus vont dans le sens attendu : un envoi tardif, un délai plus long et un gros montant font baisser la probabilité ; l'historique du client la fait monter. Les coefficients sont réduits par la régularisation : on lit leur sens et leur ordre, pas leur valeur.
- 12 devis étaient ouverts au 30 juin, classés par valeur attendue (probabilité × montant). Les statuts constatés depuis montrent que 6 ont été refusés, 5 attendent une réponse et 1 a été accepté sans qu'aucune commande ne soit saisie (DV-2026-0287) : c'est la relance la plus rentable.
- Deux listes comptent autant que le score : 23 devis non envoyés au 30 juin (41,5 M HT) et 80 devis sans réponse depuis plus de 15 jours (128,4 M HT), à relancer une dernière fois puis à clore.

## 6. Leviers et niveau de confiance

Trois leviers pour les commerciaux
1. Envoyer le devis en moins de 3 jours. C'est le levier le plus fort et il dépend entièrement de BLOK : 62 % de conversion à 0-3 jours, 6 % au-delà de 8 jours. Une alerte automatique sur tout brouillon de plus de 2 jours suffit.
2. Relancer avant J+12, en commençant par la valeur attendue. Aucune conversion n'arrive plus tard. Après J+15 : dernière relance, puis clôture.
3. Appeler avant d'envoyer un gros devis à un client sans historique : c'est le profil qui convertit le moins.

Confiance : limitée. 352 devis, un test de 88, un signal modeste (le modèle classe correctement environ 2 paires sur 3) et 8 mois d'historique. C'est un outil pour ordonner une liste de relances, pas une probabilité exacte. À réentraîner chaque mois. Le gain le plus net viendrait d'un champ « motif de refus » saisi par les commerciaux.